# The Task: Pick Pen

## 📖 Reading only — no action needed

The "Pick Pen" task is already fully implemented and installed at `~/workspace/leisaac/source/leisaac/leisaac/tasks/pick_pen/`. Skip ahead to [notebook 4](04_inference.ipynb) if you're short on time — come back here later if you want to understand how the task is defined.

## Task description

The Pick Pen task has three subtasks:

1. **Pick the pen** — robot approaches and grasps the mechanical pencil.
2. **Place on plate** — robot moves the pen to the plate and releases it.
3. **Return to rest** — robot returns to its rest position.

## How progress is detected

`tasks/pick_pen/mdp/observations.py` defines two functions used to tell whether each subtask is complete:

- `pen_grasped(...)` — true when the end effector is within `diff_threshold` of the pencil **and** the gripper joint is closed (`joint_pos[:, -1] < grasp_threshold`).
- `put_pen_to_plate(...)` — true when the pencil's x/y position falls inside the plate's `x_range`/`y_range`, the end effector is still near the pencil, and the gripper is open.

Both are wired into the environment as `subtask_terms` observations in `pick_pen_env_cfg.py`:

```python
class SubtaskCfg(ObsGroup):
    pick_pen = ObsTerm(func=mdp.pen_grasped, params={"object_cfg": SceneEntityCfg("MechanicalPencil")})
    put_pen_to_plate = ObsTerm(
        func=mdp.put_pen_to_plate,
        params={"object_cfg": SceneEntityCfg("MechanicalPencil"), "plate_cfg": SceneEntityCfg("Plate")},
    )
```

## How the episode ends

`tasks/pick_pen/mdp/terminations.py` defines `task_done(...)`, which succeeds once the pen is within range of the plate on x/y/height **and** the robot has returned to its rest pose. It's registered as the `success` termination alongside a `time_out` fallback:

```python
class TerminationsCfg:
    time_out = DoneTerm(func=mdp.time_out, time_out=True)
    success = DoneTerm(
        func=mdp.task_done,
        params={"pens_cfg": [SceneEntityCfg("MechanicalPencil")], "plate_cfg": SceneEntityCfg("Plate")},
    )
```

## Putting it together: the environment config

`pick_pen_env_cfg.py`'s `PickPenEnvCfg` combines everything: the `kitchen_with_pen` scene (previous notebook), the SO-101 robot, two cameras (wrist + front), the observation/termination groups above, and domain randomization on the pencil, plate, and camera pose — so the trained policy generalizes instead of memorizing one exact layout.

## Generating training data: the mimic environment

`pick_pen_mimic_env_cfg.py` extends the task with `PickPenMimicEnvCfg`, used only for **data generation** (not for the inference you'll run next). It breaks each demonstration into the same three subtasks (`pick_pen → put_pen_to_plate → rest robot`) so recorded trajectories can be resegmented, interpolated, and replayed with noise to synthesize more training data from a handful of human demos.

## Registering the task

Finally, `tasks/pick_pen/__init__.py` registers both environments with Gymnasium so they can be referenced by name — you'll see `LeIsaac-SO101-PickPen-v0` again in the inference notebook:

```python
gym.register(id="LeIsaac-SO101-PickPen-v0", ...)
gym.register(id="LeIsaac-SO101-PickPen-Mimic-v0", ...)
```
